# LightFM：算法、评估与推荐

先运行 `python train.py --model lightfm --loss logistic`。本 Notebook 展示选定模型，并与 ItemCF 使用相同用户和候选规则比较。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / 'src' / 'models' / 'lightfm.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommend import file_sha256, recommend_for_user

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
run_path = TABLE_DIR / 'lightfm_run.json'
if not run_path.is_file():
    raise FileNotFoundError('请先运行 python train.py --model lightfm。')
run = json.loads(run_path.read_text(encoding='utf-8'))
for name, checksum in run['input_sha256'].items():
    if file_sha256(PROJECT_ROOT / 'data' / 'processed' / name) != checksum:
        raise ValueError('预处理输入已变化，请重新训练 LightFM。')

## 1. 算法与选定参数

用户输入：用户 ID，以及性别、年龄段、职业、历史高分电影的类型和年代偏好（58 个属性）。
电影输入：电影 ID，以及类型、上映年代、热度、平滑平均评分和高分比例（31 个属性）。
身份和归一化属性两部分各占 0.5 权重，特征向量加权相加；最终表示维度由 `no_components` 决定，不随特征数量拼接增长。

训练集内部按用户时间分组约对半划分：前段构造统计特征，后段学习参数。验证/测试表示用完整训练集重建特征，验证/测试评分不参与特征统计或参数学习。
比较 ID 特征时需保持拟合标签、损失、维度、正则化及随机种子一致，逐次实验配置以运行记录为准。

采用 **logistic 损失**：拟合评分 ≥4 为 +1，1–2 分为 -1，3 分和未评分项不参与拟合。
按验证 **NDCG@10** 选参，实际已完成的参数网格读取自运行记录。可选容量实验比较 32/64/128 维各 10 轮；未完成的进度不参与最终结果比较。
训练/验证交叉熵只统计已知高低分，不含 L2 惩罚。两者采用的历史长度及正负比例不同，结合曲线趋势判断，不只比较绝对差值。

In [2]:
display(pd.DataFrame([run['selected_parameters']]))
print('Completed search grid:', {name: run['grid'][name]
                                for name in ['no_components', 'alpha', 'epochs']})
search = pd.read_csv(TABLE_DIR / 'lightfm_validation_search.csv')
columns = [name for name in ['no_components', 'alpha', 'epochs', 'train_loss',
                            'validation_loss', 'NDCG@10', 'selected']
           if name in search.columns]
display(search[columns].style.format({'train_loss': '{:.4f}', 'validation_loss': '{:.4f}',
                                     'NDCG@10': '{:.4f}'}))

loss_columns = ['train_loss', 'validation_loss']
if all(name in search.columns for name in loss_columns) and search[loss_columns].notna().all().all():
    from src.lightfm_diagnostics import plot_capacity_curves
    import matplotlib.pyplot as plt
    figure = plot_capacity_curves(search, PROJECT_ROOT / 'outputs/figures/lightfm_capacity_curves.png')
    display(figure)
    plt.close(figure)
else:
    print('当前完成的实验未记录 loss 曲线；完成容量实验后可在此展示。')


Completed search grid: {'no_components': [32], 'alpha': [1e-06], 'epochs': [5, 10]}
当前完成的实验未记录 loss 曲线；完成容量实验后可在此展示。


,loss,user_features,item_features,no_components,alpha,epochs
0,logistic,id+demographics+genre_preferences+decade_prefe...,id+genres+decade+popularity+smoothed_ratings,32,0.000001,5


,no_components,alpha,epochs,NDCG@10,selected
0,32,0.000001,5,0.0222,True
1,32,0.000001,10,0.0167,False


## 2. 与 ItemCF 比较

两种算法使用相同数据划分、评估用户和完整未评分候选目录。各指标先按用户计算再平均：
Precision@10＝命中数 /10，Recall@10＝命中数 /高分目标数，NDCG@10 同时考虑命中位置；MeanHits@10 为平均命中部数。
未评分电影不计为已知命中，但其偏好未知。LightFM 的用户向量仅由训练集学习，验证历史在测试时用于过滤已评分电影。

In [3]:
comparison = pd.read_csv(TABLE_DIR / 'comparison_metrics.csv')
comparison = comparison[comparison.algorithm.isin(['ItemCF', 'LightFM'])]
display(comparison.fillna({'loss': '-'})[['algorithm', 'loss', 'split', 'users', 'Precision@10', 'Recall@10', 'NDCG@10', 'MeanHits@10']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}',
                       'NDCG@10': '{:.4f}', 'MeanHits@10': '{:.3f}'}))

,algorithm,loss,split,users,Precision@10,Recall@10,NDCG@10,MeanHits@10
0,ItemCF,-,validation,5776,5.97%,7.89%,0.0844,0.597
1,ItemCF,-,test,5765,5.64%,7.12%,0.0773,0.564
2,LightFM,logistic,validation,5776,1.83%,1.81%,0.0222,0.183
3,LightFM,logistic,test,5765,1.76%,1.84%,0.0220,0.176


## 3. 按用户比较推荐电影

修改 `USER_ID` 后运行下面的单元格。每种算法推荐 5 部历史未评分电影，展示标题与类型。

In [4]:
USER_ID = 1
TOP_K = 5

recommendations = []
for algorithm in ['itemcf', 'lightfm']:
    result = recommend_for_user(USER_ID, stage='test', k=TOP_K, algorithm=algorithm)
    result.insert(0, 'algorithm', algorithm)
    recommendations.append(result[['algorithm', 'rank', 'title', 'genres']])
display(pd.concat(recommendations, ignore_index=True))

,algorithm,rank,title,genres
0,itemcf,1,Raiders of the Lost Ark (1981),Action|Adventure
1,itemcf,2,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,itemcf,3,"Shawshank Redemption, The (1994)",Drama
3,itemcf,4,"Silence of the Lambs, The (1991)",Drama|Thriller
4,itemcf,5,Pulp Fiction (1994),Crime|Drama
5,lightfm,1,"Shawshank Redemption, The (1994)",Drama
6,lightfm,2,"Great Escape, The (1963)",Adventure|War
7,lightfm,3,"Hustler, The (1961)",Drama
8,lightfm,4,Rear Window (1954),Mystery|Thriller
9,lightfm,5,"Godfather, The (1972)",Action|Crime|Drama


算法参考：Kula (2015), [Metadata Embeddings for User and Item Cold-start Recommendations](https://arxiv.org/abs/1507.08439)。训练使用[官方 LightFM 实现](https://making.lyst.com/lightfm/docs/lightfm.html)。